# 04 — Procesamiento Distribuido con Apache Spark (PySpark)

Apache Spark es el motor de procesamiento distribuido estándar de la industria Big Data. Procesa datos en memoria usando un modelo de ejecución lazy (DAG de transformaciones) con optimización automática via Catalyst Optimizer. En modo local usa todos los cores de la máquina; el mismo código se ejecuta sin cambios en un cluster Dataproc.

**Flujo:** CSVs locales (2017–2025) → Spark local (lazy evaluation + Catalyst) → resultados por era COVID → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Arrestos por año y era COVID — tendencia de efectividad policial | `arrests_by_year` |
| 2 | **Read**   | Crímenes por community area — comparativa geográfica entre eras | `crimes_by_district` |
| 3 | **Read**   | Tendencia mensual 2017–2025 — estacionalidad por era COVID | `monthly_trend` |
| 4 | **Update** | Añadir `is_weekend` + `covid_era` — cambio de patrones fin de semana por COVID | `weekend_analysis` |
| 5 | **Delete** | Filtrar registros inválidos — validar integridad por era | — (limpieza) |

In [1]:
# !pip install pyspark pandas-gbq

In [2]:
import os
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DoubleType, BooleanType
)

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
FOLDER     = 'Chicago_Crimes_by_Year'

# ── Definición de eras COVID ──────────────────────────────────────────────────
YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP_SPARK  = {
    2017:'PRE', 2018:'PRE', 2019:'PRE',
    2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
    2023:'POST', 2024:'POST', 2025:'POST',
}

def to_bigquery(spark_df, table_name: str, if_exists: str = 'replace') -> None:
    pdf = spark_df.toPandas()
    pdf.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({pdf.shape[0]:,} filas)')

spark = (
    SparkSession.builder
    .appName('ChicagoCrimes-COVID-CRUD')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
print(f'Spark {spark.version} — modo local')

Spark 4.1.1 — modo local


In [3]:
# ── Schema explícito (evita inferencia costosa sobre los 9 archivos) ──────────
schema = StructType([
    StructField('unique_key',           LongType(),    True),
    StructField('case_number',          StringType(),  True),
    StructField('date',                 StringType(),  True),
    StructField('block',                StringType(),  True),
    StructField('iucr',                 StringType(),  True),
    StructField('primary_type',         StringType(),  True),
    StructField('description',          StringType(),  True),
    StructField('location_description', StringType(),  True),
    StructField('arrest',               StringType(),  True),
    StructField('domestic',             StringType(),  True),
    StructField('beat',                 IntegerType(), True),
    StructField('district',             IntegerType(), True),
    StructField('ward',                 IntegerType(), True),
    StructField('community_area',       IntegerType(), True),
    StructField('fbi_code',             StringType(),  True),
    StructField('x_coordinate',         LongType(),    True),
    StructField('y_coordinate',         LongType(),    True),
    StructField('year',                 IntegerType(), True),
    StructField('updated_on',           StringType(),  True),
    StructField('latitude',             DoubleType(),  True),
    StructField('longitude',            DoubleType(),  True),
    StructField('location',             StringType(),  True),
])

# Solo los 9 años del análisis COVID
files = [os.path.join(FOLDER, f'Chicago_Crimes_{y}.csv') for y in YEARS_ANALYSIS]

# Expresión CASE WHEN para covid_era en SQL
era_expr = F.when(F.col('year') <= 2019, 'PRE') \
            .when(F.col('year') <= 2022, 'DURANTE') \
            .otherwise('POST')

t0 = time.time()
df_raw = (
    spark.read
         .option('header', 'true')
         .option('nullValue', '')
         .schema(schema)
         .csv(files)
)

df = (
    df_raw
    .withColumn('date',       F.to_timestamp('date'))
    .withColumn('updated_on', F.to_timestamp('updated_on'))
    .withColumn('arrest',     F.lower(F.col('arrest')) == 'true')
    .withColumn('domestic',   F.lower(F.col('domestic')) == 'true')
    .withColumn('covid_era',  era_expr)
)
df.cache()
total = df.count()
print(f'Total de registros: {total:,}  ({time.time()-t0:.1f}s)')
print(f'Años cargados:      {YEARS_ANALYSIS}')

df.createOrReplaceTempView('crimes')
print('Vista SQL "crimes" disponible.')

Total de registros: 2,072,943  (10.5s)
Años cargados:      [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


Vista SQL "crimes" disponible.


---
## CRUD 1 — CREATE: Arrestos por año y era COVID

**Operación:** Se crea la tabla `arrests_by_year` que cruza año y era COVID, mostrando la evolución de la efectividad policial (tasa de arresto) a lo largo del período 2017–2025.

**Hipótesis:** La tasa de arresto debería caer en 2020 por las restricciones operativas del CPD (protocolos COVID, reducción de personal en campo), y recuperarse gradualmente en la era POST.

**Justificación Big Data:** Spark SQL es la interfaz más eficiente para aggregations complejas — el Catalyst Optimizer genera un plan físico optimizado (predicate pushdown, partial aggregation por partición) que evita barridos innecesarios de datos.

In [4]:
arrests_by_year = spark.sql("""
    SELECT
        year,
        covid_era,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct
    FROM crimes
    WHERE year IS NOT NULL
    GROUP BY year, covid_era
    ORDER BY year
""")

arrests_by_year.show(30)

# Resumen por era
print('Tasa de arresto promedio por era:')
arrests_by_year.groupBy('covid_era') \
    .agg(
        F.round(F.sum('total_crimes'), 0).alias('total_crimes'),
        F.round(F.avg('arrest_rate_pct'), 2).alias('avg_arrest_rate_pct'),
    ) \
    .orderBy('covid_era') \
    .show()

to_bigquery(arrests_by_year, 'arrests_by_year')

+----+---------+------------+-------------+---------------+
|year|covid_era|total_crimes|total_arrests|arrest_rate_pct|
+----+---------+------------+-------------+---------------+
|2017|      PRE|      269214|        52670|          19.56|
|2018|      PRE|      269070|        53901|          20.03|
|2019|      PRE|      261555|        56256|          21.51|
|2020|  DURANTE|      212522|        34158|          16.07|
|2021|  DURANTE|      209406|        26558|          12.68|
|2022|  DURANTE|      239655|        28074|          11.71|
|2023|     POST|      262756|        31843|          12.12|
|2024|     POST|      256305|        34502|          13.46|
|2025|     POST|       92460|        15117|          16.35|
+----+---------+------------+-------------+---------------+

Tasa de arresto promedio por era:


+---------+------------+-------------------+
|covid_era|total_crimes|avg_arrest_rate_pct|
+---------+------------+-------------------+
|  DURANTE|      661583|              13.49|
|     POST|      611521|              13.98|
|      PRE|      799839|              20.37|
+---------+------------+-------------------+



C:\Users\Usuario\AppData\Local\Temp\ipykernel_26260\4268181033.py:25: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


  → BigQuery: my-first-project-492901.chicago_crimes_results.arrests_by_year  (9 filas)


---
## CRUD 2 — READ: Crímenes por community area — comparativa geográfica entre eras

**Operación:** Análisis geográfico a nivel de las 77 áreas comunitarias de Chicago, calculando volumen y tasa de arresto para cada era COVID. Permite identificar si el COVID redistribuyó geográficamente la criminalidad (ej. del Loop hacia zonas residenciales).

**Hipótesis:** Las áreas del centro (Loop, Near North Side, Lincoln Park — community areas 8, 32, 7) deberían mostrar la mayor caída en DURANTE por cierre de comercios y turismo. Las áreas residenciales de alta densidad (Austin, Humboldt Park) podrían mantenerse o subir.

**Justificación Big Data:** La agregación sobre 77 claves geográficas con múltiples métricas simultáneas se ejecuta en un solo pasaje sobre los datos (one-pass aggregation) gracias al motor vectorizado de Spark.

In [5]:
crimes_by_area = spark.sql("""
    SELECT
        community_area                              AS district,
        covid_era,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct,
        COUNT(DISTINCT primary_type)                AS distinct_crime_types,
        SUM(CAST(domestic AS INT))                  AS domestic_incidents
    FROM crimes
    WHERE community_area IS NOT NULL
    GROUP BY community_area, covid_era
    ORDER BY community_area, covid_era
""")

# Top 5 áreas por era
print('Top 5 áreas comunitarias con más crímenes por era:')
for era in ['PRE', 'DURANTE', 'POST']:
    print(f'\n  Era {era}:')
    crimes_by_area.filter(F.col('covid_era') == era) \
        .orderBy(F.col('total_crimes').desc()) \
        .select('district', 'total_crimes', 'arrest_rate_pct') \
        .show(5)

to_bigquery(crimes_by_area, 'crimes_by_district')

Top 5 áreas comunitarias con más crímenes por era:

  Era PRE:


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|      25|       45474|          25.36|
|       8|       38067|          16.99|
|      32|       32144|          15.74|
|      28|       27944|          15.61|
|      29|       27611|          38.92|
+--------+------------+---------------+
only showing top 5 rows

  Era DURANTE:


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|      25|       36802|          16.79|
|       8|       26350|           13.4|
|      43|       23617|          10.76|
|      28|       23000|          10.21|
|      29|       20688|          24.04|
+--------+------------+---------------+
only showing top 5 rows

  Era POST:


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|      25|       30504|          16.97|
|       8|       26511|           15.5|
|      28|       25099|           9.92|
|      32|       21277|          20.55|
|      43|       20405|          10.31|
+--------+------------+---------------+
only showing top 5 rows


C:\Users\Usuario\AppData\Local\Temp\ipykernel_26260\4268181033.py:25: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.crimes_by_district  (231 filas)


---
## CRUD 3 — READ: Tendencia mensual 2017–2025 — estacionalidad por era COVID

**Operación:** Serie de tiempo que desglosa crímenes mes a mes durante los 9 años del análisis, añadiendo la era COVID de cada punto. Permite comparar los patrones estacionales (verano vs invierno) antes, durante y después del COVID.

**Hipótesis:** El pico estacional de verano (julio–agosto) debería desaparecer o reducirse drásticamente en 2020 por el toque de queda. En 2021–2022, la recuperación debería ser gradual. En POST, el patrón estacional debería retomar la forma pre-pandémica.

**Justificación Big Data:** La extracción de componentes temporales (`MONTH`, `YEAR`) y la agregación sobre ellos es una operación que Spark ejecuta eficientemente gracias al particionamiento en memoria por columna de fechas.

In [6]:
monthly_trend = spark.sql("""
    SELECT
        year,
        covid_era,
        MONTH(date)                                 AS month,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct
    FROM crimes
    WHERE date IS NOT NULL AND year IS NOT NULL
    GROUP BY year, covid_era, MONTH(date)
    ORDER BY year, month
""")

print(f'Puntos temporales (año×mes): {monthly_trend.count()}')

# Estacionalidad promedio de verano vs invierno por era
monthly_trend.createOrReplaceTempView('monthly')
seasonal = spark.sql("""
    SELECT
        covid_era,
        CASE WHEN month IN (6,7,8) THEN 'VERANO' ELSE 'RESTO' END AS season,
        ROUND(AVG(total_crimes), 0) AS avg_monthly_crimes
    FROM monthly
    GROUP BY covid_era, CASE WHEN month IN (6,7,8) THEN 'VERANO' ELSE 'RESTO' END
    ORDER BY covid_era, season
""")
print('Criminalidad promedio mensual — verano vs resto por era:')
seasonal.show()

to_bigquery(monthly_trend, 'monthly_trend')

Puntos temporales (año×mes): 103
Criminalidad promedio mensual — verano vs resto por era:


+---------+------+------------------+
|covid_era|season|avg_monthly_crimes|
+---------+------+------------------+
|  DURANTE| RESTO|           17855.0|
|  DURANTE|VERANO|           19944.0|
|     POST| RESTO|           19353.0|
|     POST|VERANO|           21006.0|
|      PRE| RESTO|           21416.0|
|      PRE|VERANO|           24623.0|
+---------+------+------------------+



C:\Users\Usuario\AppData\Local\Temp\ipykernel_26260\4268181033.py:25: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.monthly_trend  (103 filas)


---
## CRUD 4 — UPDATE: Añadir `is_weekend` + análisis por día de semana cruzado con era COVID

**Operación:** Se enriquece el dataset con `is_weekend` y `day_name`. Se compara si el COVID cambió la distribución de crímenes entre días hábiles y fin de semana — efecto esperado: con el cierre de negocios, la distinción viernes-noche/sábado podría diluirse.

**Hipótesis:** En PRE, los fines de semana concentrarían más crímenes (mayor actividad nocturna). Durante el confinamiento (DURANTE), la diferencia entre semana y fin de semana debería reducirse al homogeneizarse los patrones de movilidad.

**Justificación Big Data:** La función `dayofweek()` de Spark opera a nivel de columna vectorizada (no fila a fila como Python UDFs), procesando los 2M timestamps en paralelo sin overhead de serialización.

In [7]:
# dayofweek: 1=Domingo, 7=Sábado
df_enriched = (
    df
    .withColumn('is_weekend', F.dayofweek('date').isin([1, 7]))
    .withColumn('day_name',   F.date_format('date', 'EEEE'))
)
df_enriched.createOrReplaceTempView('crimes')

weekend_analysis = spark.sql("""
    SELECT
        covid_era,
        is_weekend,
        COUNT(*)                                            AS total_crimes,
        SUM(CAST(arrest AS INT))                            AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)            AS arrest_rate_pct,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY covid_era), 2) AS pct_of_era_total
    FROM crimes
    WHERE is_weekend IS NOT NULL
    GROUP BY covid_era, is_weekend
    ORDER BY covid_era, is_weekend DESC
""")

print('Fin de semana vs días de semana por era COVID:')
weekend_analysis.show()

to_bigquery(weekend_analysis, 'weekend_analysis')

Fin de semana vs días de semana por era COVID:


+---------+----------+------------+-------------+---------------+----------------+
|covid_era|is_weekend|total_crimes|total_arrests|arrest_rate_pct|pct_of_era_total|
+---------+----------+------------+-------------+---------------+----------------+
|  DURANTE|      true|      187299|        25687|          13.71|           28.31|
|  DURANTE|     false|      474284|        63103|           13.3|           71.69|
|     POST|      true|      173790|        22997|          13.23|           28.42|
|     POST|     false|      437731|        58465|          13.36|           71.58|
|      PRE|      true|      222090|        46171|          20.79|           27.77|
|      PRE|     false|      577749|       116656|          20.19|           72.23|
+---------+----------+------------+-------------+---------------+----------------+



C:\Users\Usuario\AppData\Local\Temp\ipykernel_26260\4268181033.py:25: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.weekend_analysis  (6 filas)


---
## CRUD 5 — DELETE: Filtrar registros inválidos y verificar integridad por era

**Operación:** Se eliminan registros sin `primary_type` o sin `unique_key`, y se reporta cuántos registros inválidos hay por era. Una proporción mayor en DURANTE podría indicar sobrecarga operativa del sistema CPD durante la pandemia (reportes incompletos, sobrecarga administrativa).

**Justificación Big Data:** El filtrado por condiciones nulas en Spark aprovecha predicate pushdown — si los datos estuviesen en Parquet, Spark solo leería los row groups que potencialmente contienen nulos, sin leer el archivo completo.

In [8]:
invalid = df_enriched.filter(
    F.col('primary_type').isNull() |
    (F.trim(F.col('primary_type')) == '') |
    F.col('unique_key').isNull()
)

# Registros inválidos por era
invalid_by_era = (
    invalid.groupBy('covid_era')
           .count()
           .withColumnRenamed('count', 'invalid_records')
)
total_by_era = (
    df_enriched.groupBy('covid_era')
               .count()
               .withColumnRenamed('count', 'total')
)
report = total_by_era.join(invalid_by_era, on='covid_era', how='left').fillna(0)
report = report.withColumn('pct_invalid',
    F.round(F.col('invalid_records') / F.col('total') * 100, 4))

print(f'Registros totales:   {total:,}')
print('Registros inválidos por era:')
report.orderBy('covid_era').show()

df_valid = df_enriched.filter(
    F.col('primary_type').isNotNull() &
    (F.trim(F.col('primary_type')) != '') &
    F.col('unique_key').isNotNull()
)
n_valid = df_valid.count()
print(f'Registros válidos:   {n_valid:,}')

Registros totales:   2,072,943
Registros inválidos por era:


+---------+------+---------------+-----------+
|covid_era| total|invalid_records|pct_invalid|
+---------+------+---------------+-----------+
|  DURANTE|661583|              0|        0.0|
|     POST|611521|              0|        0.0|
|      PRE|799839|              0|        0.0|
+---------+------+---------------+-----------+



Registros válidos:   2,072,943


In [9]:
spark.stop()
print('SparkSession cerrada. Resultados guardados en BigQuery.')

SparkSession cerrada. Resultados guardados en BigQuery.
